# Adaptateur Assetto Corsa pour le pilote optionnel

Réutilise le départ fixe CSP, le frein de surveillance et les contrôles du pilote historique. Réserve vJoy lors du premier reset.

In [ ]:
def frame_to_image84(frame):
    if frame.ndim==3:
        # La capture couleur du plugin est en BGR.
        gray=cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)
    elif frame.ndim==2:
        gray=frame
    else:
        raise ValueError('Format camera invalide : '+str(frame.shape))
    gray=cv2.resize(gray,(84,84),interpolation=cv2.INTER_AREA)
    return np.ascontiguousarray(gray[np.newaxis,:,:],dtype=np.uint8)

In [ ]:
import numpy as np, pandas as pd
from scipy.spatial import cKDTree

class TrackGeometry:
    def __init__(self,path):
        df=pd.read_csv(path)
        required=['left_border_x','left_border_y','right_border_x','right_border_y']
        if not set(required).issubset(df.columns):
            raise ValueError('CSV geometrie non compatible, colonnes attendues: '+str(required))
        left=df[['left_border_x','left_border_y']].to_numpy(dtype=np.float64)
        right=df[['right_border_x','right_border_y']].to_numpy(dtype=np.float64)
        self.center=(left+right)*.5
        self.width=np.linalg.norm(left-right,axis=1)
        self.delta=np.roll(self.center,-1,axis=0)-self.center
        self.segment=np.linalg.norm(self.delta,axis=1)
        if (self.segment>.001).sum()<10:
            raise ValueError('CSV de circuit invalide: pas assez de points')
        self.arc=np.concatenate(([0.],np.cumsum(self.segment)))
        self.length=float(self.arc[-1])
        self.tree=cKDTree(self.center)
    def project(self,raw):
        xy=np.asarray([float(raw['world_position_x']),float(raw['world_position_y'])],dtype=np.float64)
        _,j=self.tree.query(xy)
        j=int(j)
        forward=self.delta[j]/max(float(self.segment[j]),1e-8)
        signed=float(forward[0]*(xy[1]-self.center[j,1])-forward[1]*(xy[0]-self.center[j,0]))
        heading=float(np.arctan2(forward[1],forward[0]))
        return float(self.arc[j]),signed,float(self.width[j]),heading

def numerical_state(raw,track):
    arc,offset,width,heading=track.project(raw)
    diff=(float(raw.get('yaw',0))-heading+np.pi)%(2*np.pi)-np.pi
    vec=[float(raw.get('speed',0))/100.,float(raw.get('accelX',0))/3.,
         float(raw.get('accelY',0))/3.,float(raw.get('angular_velocity_y',0))/4.,
         float(np.clip(offset/max(width/2.,1.),-3,3)),np.sin(diff),np.cos(diff),
         float(raw.get('accStatus',0)),float(raw.get('brakeStatus',0))]
    return np.clip(np.asarray(vec,dtype=np.float32),-5,5),arc,offset,width

def find_track_file(track_name):
    # Les fichiers .pkl sont des grilles d'occupation; ici la recompense
    # utilise les CSV bord gauche/bord droit du depot assetto_corsa_gym.
    name=str(track_name)
    path=GYM_ROOT/'AssettoCorsaConfigs'/'tracks'/(name+'.csv')
    if not path.is_file():
        raise FileNotFoundError('CSV de geometrie introuvable : '+str(path))
    return path

In [ ]:
import ctypes, threading, time, numpy as np
from pathlib import Path

class VJoySafe:
    """Pilote local Windows deja valide dans 04_Test_Volant_vJoy/05_Test_Pedales."""
    X=0x30; Y=0x31; Z=0x32
    def __init__(self,device_id=1,watchdog_sec=.7):
        dll_path=Path(r'C:\Program Files\vJoy\x64\vJoyInterface.dll')
        if not dll_path.exists(): raise FileNotFoundError(str(dll_path))
        self.dll=ctypes.CDLL(str(dll_path)); self.device_id=int(device_id)
        self.dll.GetVJDStatus.argtypes=[ctypes.c_uint]
        self.dll.GetVJDStatus.restype=ctypes.c_int
        self.dll.AcquireVJD.argtypes=[ctypes.c_uint]
        self.dll.AcquireVJD.restype=ctypes.c_int
        self.dll.SetAxis.argtypes=[ctypes.c_long,ctypes.c_uint,ctypes.c_uint]
        self.dll.SetAxis.restype=ctypes.c_int
        self.dll.RelinquishVJD.argtypes=[ctypes.c_uint]
        self.dll.RelinquishVJD.restype=None
        status=self.dll.GetVJDStatus(self.device_id)
        if status!=1: raise RuntimeError('vJoy peripherique %d indisponible, statut=%d'%(self.device_id,status))
        if not self.dll.AcquireVJD(self.device_id): raise RuntimeError('Echec AcquireVJD')
        self.lock=threading.Lock(); self.active=True
        self.last_action=time.monotonic(); self.watchdog_sec=float(watchdog_sec)
        self.last_command={'steer_cmd':0.0, 'throttle_cmd':0.0, 'brake_cmd':1.0}
        self._set_axes(16384,0,32767)  # frein de securite lors de la prise de controle
        self.thread=threading.Thread(target=self._watchdog,daemon=True)
        self.thread.start()
    def _set_axes(self,x,y,z):
        with self.lock:
            for axis,value in ((self.X,x),(self.Y,y),(self.Z,z)):
                if not self.dll.SetAxis(int(value),self.device_id,axis):
                    raise RuntimeError('Echec SetAxis vJoy sur '+hex(axis))
    def action(self,a):
        a=np.asarray(a,dtype=np.float32).reshape(-1)
        if a.shape!=(3,) or not np.isfinite(a).all(): raise ValueError('Action PPO invalide')
        steer=float(np.clip(a[0],-1.,1.))
        gas=float(np.clip(a[1],0.,1.))
        brake=float(np.clip(a[2],0.,1.))
        # Politique PPO au depart aleatoire : accelerateur et frein peuvent etre positifs.
        # Une faible demande de frein ne doit plus annuler un fort accelerateur.
        # Commandes longitudinales mutuellement exclusives : retenir la plus forte.
        if gas >= brake:
            brake = 0.0
        else:
            gas = 0.0
        self._set_axes(round((steer+1)*16383.5),round(gas*32767),round(brake*32767))
        # Ces valeurs sont les commandes *effectivement envoyees* a vJoy.
        self.last_command = {'steer_cmd':steer, 'throttle_cmd':gas, 'brake_cmd':brake}
        self.last_action=time.monotonic()
    def hold(self):
        if self.active:
            self._set_axes(16384,0,32767)
            self.last_action=time.monotonic()
    def _watchdog(self):
        while self.active:
            if time.monotonic()-self.last_action>self.watchdog_sec:
                try: self.hold()
                except Exception: pass
            time.sleep(.06)
    def close(self):
        if not self.active: return
        try:self.hold()
        finally:
            self.active=False
            self.thread.join(timeout=.3)
            self.dll.RelinquishVJD(self.device_id)

In [ ]:
# Cellule de pont entre Jupyter et l'application CSP Lua OUARTICheckpoint.
import json, os, time, uuid, math
from pathlib import Path

BRIDGE_DIR = Path.home() / 'Documents' / 'Assetto Corsa' / 'OUARTI_checkpoint_v3'
ANCHOR_FILE = BRIDGE_DIR / 'anchor.json'

class CheckpointBridge:
    def __init__(self, directory=BRIDGE_DIR):
        self.directory = Path(directory)
        self.command = self.directory / 'command.txt'
        self.response = self.directory / 'status.txt'

    def request(self, command, timeout=8.0):
        if command not in ('PING', 'STATUS', 'SAVE', 'LOAD', 'CAPABILITIES'):
            raise ValueError('Commande CSP inconnue: ' + command)
        if not self.directory.is_dir():
            raise RuntimeError('Application CSP OUARTICheckpoint absente : ' + str(self.directory))
        token = uuid.uuid4().hex
        temporary = self.directory / ('command_' + token + '.tmp')
        temporary.write_text(token + '|' + command, encoding='ascii')
        for _attempt in range(40):
            try:
                os.replace(str(temporary), str(self.command))
                break
            except PermissionError:
                if _attempt == 39:
                    temporary.unlink(missing_ok=True)
                    raise
                time.sleep(0.025)
        deadline = time.monotonic() + float(timeout)
        last = ''
        while time.monotonic() < deadline:
            try:
                last = self.response.read_text(encoding='utf8').strip()
            except (FileNotFoundError, UnicodeError, PermissionError):
                time.sleep(0.05)
                continue
            parts = last.split('|', 2)
            if len(parts) == 3 and parts[0] == token:
                if parts[1] == 'ERROR':
                    raise RuntimeError('CSP checkpoint: ' + parts[2])
                if parts[1] == 'PENDING':
                    time.sleep(0.05)
                    continue
                return parts[1], parts[2]
            time.sleep(0.05)
        raise TimeoutError('Pas de reponse CSP pour ' + command + '. Dernier statut: ' + last[:140])

    def capabilities(self):
        code, detail = self.request('CAPABILITIES')
        if code != 'CAPABILITIES':
            raise RuntimeError('Reponse CAPABILITIES inattendue: ' + code + ' ' + detail)
        fields = {}
        for field in detail.split(';'):
            key, separator, value = field.strip().partition('=')
            if separator:
                fields[key.strip()] = value.strip()
        for key in ('offline', 'full_snapshot', 'position_fallback'):
            if fields.get(key) not in ('true', 'false'):
                raise RuntimeError('Capacite CSP absente ou invalide: ' + key + ': ' + detail)
        return {key: value == 'true' for key, value in fields.items()
                if key in ('offline', 'full_snapshot', 'position_fallback')}

    def assert_supported(self):
        caps = self.capabilities()
        if not caps['offline']:
            raise RuntimeError('Checkpoint reserve a une session locale hors ligne.')
        if not (caps['full_snapshot'] or caps['position_fallback']):
            raise RuntimeError(
                'Checkpoint indisponible: CSP ne fournit ni snapshot complet ni '
                'repositionnement autorise dans cette session. La connexion au pont fonctionne. '
                'Verifier la version CSP dans Content Manager > Settings > Custom Shaders Patch '
                '> About & Updates. Sur le PC examine le 16/11/2026, CSP 0.1.79 est installe '
                'et son SDK ne declare pas saveCarStateAsync/loadCarState. '
                'Apres mise a jour compatible avec sensors_par, relancer le jeu et refaire '
                'PING puis CAPABILITIES. Si les deux capacites restent false, ne pas lancer PPO. '
                'Le fallback position depend aussi de physics.allowed(); '
                'une mise a jour seule ne garantit pas son activation.')
        return caps

    def assert_saved(self):
        code, detail = self.request('STATUS')
        if code != 'SAVED':
            raise RuntimeError('Checkpoint non enregistre dans cette session CSP: ' + detail
                               + '. Executer la cellule de sauvegarde avec ACTIVER_SAUVEGARDE_CHECKPOINT=True, '
                               'voiture immobile, puis remettre False. Refaire SAVE apres chaque redemarrage du jeu.')
        return detail

    def load(self):
        code, detail = self.request('LOAD', timeout=8)
        if code != 'LOAD_REQUESTED':
            raise RuntimeError('CSP n a pas confirme la commande LOAD: ' + code + ' ' + detail)


def telemetry_xy(raw):
    return float(raw['world_position_x']), float(raw['world_position_y'])


def distance_from_anchor(raw, anchor):
    x, y = telemetry_xy(raw)
    xa, ya = float(anchor['x']), float(anchor['y'])
    distance = math.hypot(x - xa, y - ya)
    angle_error = abs((float(raw['yaw']) - float(anchor['yaw']) + math.pi) % (2 * math.pi) - math.pi)
    return distance, angle_error


def make_anchor_from_raw(raw, track, car):
    x, y = telemetry_xy(raw)
    if float(raw.get('speed', 999)) > 0.3:
        raise RuntimeError('Immobiliser la voiture avant sauvegarde (<1 km/h).')
    return {'track': track, 'car': car, 'x': x, 'y': y,
            'yaw': float(raw['yaw']), 'created_at': time.time(),
            'note': 'Checkpoint CSP en memoire, non reutilisable apres redemarrage du jeu.'}

In [ ]:
import os, json, time, numpy as np, gymnasium as gym
from gymnasium import spaces

class LiveAssettoVisionEnv(gym.Env):
    """1 instance = 1 jeu local, 1 connexion UDP, 1 vJoy. Jamais de simulation sans jeu."""
    metadata={'render_modes':[]}
    def __init__(self,expected_track,expected_car,episode_seconds=45,
                 action_interval=.12,progress_sign=1,log_file=None,curriculum_stage=1,offtrack_penalty=100.0):
        super().__init__()
        self.offtrack_penalty=float(offtrack_penalty)
        if not np.isfinite(self.offtrack_penalty) or self.offtrack_penalty <= 0:
            raise ValueError('La penalite de sortie doit etre un nombre positif fini.')
        self.expected_track=str(expected_track)
        self.expected_car=str(expected_car)
        self.curriculum_stage=int(curriculum_stage)
        self.goals={1:10.0,2:30.0,3:100.0,4:None}
        if self.curriculum_stage not in self.goals:
            raise ValueError('Etape du programme: 1, 2, 3 ou 4.')
        self.goal_m=OBJECTIF_DISTANCE_M
        self._start_lap=0
        self._lap_crossed=False
        self._lap_invalid=False
        self.episode_progress=0.0
        self.rollout_remaining=None
        self.progress_sign=int(progress_sign)
        if self.progress_sign not in (-1,1): raise ValueError('progress_sign doit valoir -1 ou 1')
        self.episode_seconds=float(episode_seconds)
        self.action_interval=float(action_interval)
        self.action_space=spaces.Box(low=np.array([-1,0,0],dtype=np.float32),
                                     high=np.ones(3,dtype=np.float32),dtype=np.float32)
        self.observation_space=spaces.Dict({
            'image':spaces.Box(low=0,high=255,shape=(1,84,84),dtype=np.uint8),
            'state':spaces.Box(low=-5.,high=5.,shape=(9,),dtype=np.float32)})
        config=management('get_config'); sess=management('get_static_info')
        track=sess.get('TrackName','')
        if sess.get('TrackConfiguration'): track+='-'+str(sess['TrackConfiguration'])
        car=str(sess.get('CarName',''))
        if track!=self.expected_track or car!=self.expected_car:
            raise RuntimeError('Mauvaise session : '+track+'/'+car+' attendu '+self.expected_track+'/'+self.expected_car)
        if bool(config.get('vjoy_executed_by_server')):
            raise RuntimeError('Ce mode utilise vJoy local: laisser vjoy_executed_by_server=False.')
        if not config.get('screen_capture_enable'):
            raise RuntimeError('Capture camera desactivee dans le plugin')
        if sess.get('autoShifterOn') is False:
            raise RuntimeError('Activer la boite automatique dans Assetto Corsa avant PPO.')
        self.track=TrackGeometry(find_track_file(track))
        self.camera=CameraReader(config)
        self.sock=None; self.controller=None
        self.raw=None; self.arc=None;self.damage=0.;self.lap=0
        self.t0=0.;self.last_obs=None;self.idle_since=0.;self.episode_index=0;self.steps=0
        self.last_action=np.zeros(3,np.float32)
        self.log_file=Path(log_file) if log_file else None
        self.bridge=CheckpointBridge()
        self.bridge.request('PING', timeout=5)
        self.bridge.assert_supported()
        self.bridge.assert_saved()
        if not ANCHOR_FILE.is_file():
            raise RuntimeError('Point de depart inconnu : utiliser la cellule SAUVEGARDER CHECKPOINT avant PPO.')
        self.anchor=json.loads(ANCHOR_FILE.read_text(encoding='utf8'))
        if self.anchor.get('track')!=track or self.anchor.get('car')!=car:
            raise RuntimeError('Checkpoint sauvegarde pour une autre voiture ou un autre circuit.')
        if self.log_file: self.log_file.parent.mkdir(parents=True,exist_ok=True)
        print('Config controle local: vJoy=local, plugin',config.get('sampling_freq'),'Hz, camera',config['color_mode'])
    def begin_rollout(self, steps):
        self.rollout_remaining = int(steps)
        if self.rollout_remaining <= 0:
            raise ValueError('Nombre de pas du lot invalide')

    def safe_stop(self):
        if self.controller is not None: self.controller.hold()
    def _connect(self):
        if self.controller is None:
            self.controller=VJoySafe(device_id=1)
        if self.sock is None:
            self.sock=open_telemetry()
    def _drain_pending_telemetry(self, max_packets=4096, max_seconds=0.12):
        """Vide les vieux datagrammes UDP sans bloquer ni perdre le timeout initial.

        A appeler apres le reset et avant la lecture de chaque observation :
        pendant time.sleep(), le buffer UDP conserve des mesures deja perimees.
        """
        if self.sock is None:
            return 0
        timeout_before = self.sock.gettimeout()
        removed = 0
        deadline = time.monotonic() + float(max_seconds)
        try:
            self.sock.setblocking(False)
            while removed < max_packets and time.monotonic() < deadline:
                try:
                    self.sock.recv(2**18)
                    removed += 1
                except (BlockingIOError, InterruptedError):
                    break
        finally:
            self.sock.settimeout(timeout_before)
        return removed

    def _observe(self, after_steps=None):
        """Obtient une image et une telemetrie recentes, avec delai controle.

        Si elles ne correspondent pas, on reprend une PAIRE fraiche plutot
        que de conserver la telemetrie d'un essai precedent.
        """
        deadline = time.monotonic() + 4.0
        last_error = 'Aucune paire image/telemetrie valide'
        while time.monotonic() < deadline:
            self._drain_pending_telemetry()
            remaining = deadline - time.monotonic()
            if remaining <= 0:
                break
            try:
                raw = next_packet(
                    self.sock,
                    timeout=min(0.8, remaining),
                    after_steps=after_steps,
                )
                remaining = deadline - time.monotonic()
                if remaining <= 0:
                    break
                seq, stamp, frame = self.camera.read(
                    timeout=min(0.8, remaining),
                )
            except (TimeoutError, RuntimeError, OSError, ConnectionError) as exc:
                last_error = str(exc)
                continue

            packet_stamp = float(raw['timestamp_unix'] if 'timestamp_unix' in raw else raw['_received_unix'])
            gap = abs(float(stamp) - packet_stamp)
            if not np.isfinite(packet_stamp) or abs(time.time()-packet_stamp) > .55 or gap > .40:
                now = time.time()
                last_error = (
                    'Camera/telemetrie non synchronisees: ecart %.3f s '
                    '(age image %.3f s, age telemetrie %.3f s)'
                    % (gap, now - float(stamp), now - packet_stamp)
                )
                continue

            verifier_fenetre_jeu()
            self.image_guidage = frame.copy()
            state, arc, lateral, width = numerical_state(raw, self.track)
            obs = {
                'image': frame_to_image84(frame),
                'state': state,
            }
            return raw, obs, arc, lateral, width, seq, gap
        raise TimeoutError('Impossible de synchroniser les capteurs: ' + last_error)
    def reset(self,*,seed=None,options=None):
        super().reset(seed=seed)
        self._connect()
        self.safe_stop()
        self.bridge.assert_saved()
        # Ici on RESTAURE un snapshot CSP, pas un reset local sans retour au depart.
        self.bridge.load()
        time.sleep(0.45)
        self._drain_pending_telemetry()
        last_err='Pas de telemetrie valide apres restauration'
        deadline=time.monotonic()+10.0
        while time.monotonic()<deadline:
            try:
                raw,o,arc,lat,width,seq,gap=self._observe()
                distance,angle=distance_from_anchor(raw,self.anchor)
                speed=float(raw.get('speed',999.))
                if distance>1.5 or angle>0.30 or speed>0.75:
                    last_err=('Checkpoint non restaure: position %.2f m, orientation %.2f rad, '
                              'vitesse %.2f m/s' % (distance,angle,speed))
                    time.sleep(0.08)
                    continue
                break
            except (TimeoutError,RuntimeError,OSError,ConnectionError,ValueError) as e:
                last_err=str(e)
        else:
            self.safe_stop()
            raise RuntimeError('RESET FIXE REFUSE : '+last_err+'. Entrainer = False.')
        self.raw=raw;self.last_obs=o;self.arc=arc
        self.damage=float(raw.get('damage_total',0))
        self.lap=int(raw.get('LapCount',0))
        self._start_lap=self.lap
        self._lap_crossed=False
        self._lap_invalid=False
        if self.curriculum_stage==4:
            if 'LapCount' not in raw or 'LapInvalidated' not in raw:
                raise RuntimeError('Tour complet : LapCount et LapInvalidated requis.')
            if bool(raw['LapInvalidated']):
                raise RuntimeError('Le checkpoint contient deja un tour invalide. Recreer un depart valide dans une nouvelle session.')
        self.idle_since=time.monotonic()
        self.t0=time.monotonic();self.steps=0;self.episode_index+=1
        self.last_action=np.zeros(3,np.float32)
        self.episode_progress=0.0
        print('Reset fixe confirme: %.2f m / %.2f rad / %.2f m/s' % (distance,angle,speed))
        return o,{'camera_gap_s':gap,'camera_sequence':seq,'track':self.expected_track,
                  'reset_verified':True,'reset_distance_m':distance,'reset_yaw_error_rad':angle,
                  'curriculum_stage':self.curriculum_stage,'goal_distance_m':self.goal_m}
    def step(self,action):
        if self.sock is None or self.raw is None:
            raise RuntimeError('reset() doit etre appele avant step()')
        if self.rollout_remaining is not None:
            self.rollout_remaining -= 1
        rollout_cut = self.rollout_remaining == 0
        action=np.asarray(action,dtype=np.float32)
        try:
            verifier_fenetre_jeu()
            self.controller.action(action)
            time.sleep(self.action_interval)
            raw,o,arc,lateral,width,seq,gap=self._observe(after_steps=self.raw.get('steps'))
            dt=float(raw['timestamp_unix'] if 'timestamp_unix' in raw else raw['_received_unix'])-float(self.raw['timestamp_unix'] if 'timestamp_unix' in self.raw else self.raw['_received_unix'])
            if not .005<dt<1.0: raise RuntimeError('Delta temps telemetrie invalide: %s'%dt)
            move=(arc-self.arc+self.track.length/2)%self.track.length-self.track.length/2
            move*=self.progress_sign
            if abs(move)>max(4.,100.*dt*1.7):
                # Collision, teleportation, mauvaise projection ou horodatage : episode invalide.
                raise RuntimeError('Saut de position non physique (%.1f m sur %.3f s)'%(move,dt))
            tyres=int(raw.get('numberOfTyresOut',0))
            offtrack=bool(tyres>2 or abs(lateral)>width*.5)
            new_damage=float(raw.get('damage_total',0))
            collision=bool(new_damage>self.damage+.02)
            new_lap=int(raw.get('LapCount',0))
            if self.curriculum_stage==4:
                if 'LapCount' not in raw or 'LapInvalidated' not in raw:
                    raise RuntimeError('Compteurs de tour absents.')
                if new_lap<self.lap or new_lap>self.lap+1:
                    raise RuntimeError('Compteur de tours incoherent.')
                self._lap_crossed=self._lap_crossed or new_lap>self.lap
                self._lap_invalid=self._lap_invalid or bool(raw['LapInvalidated'])
            lap_finished=False
            now=time.monotonic(); speed=float(raw.get('speed',0))
            if speed>1.:self.idle_since=now
            idle=bool(now-self.idle_since>6.)
            wrong_way=bool(move<-.25)
            # Curriculum sans imitation : progres geometrie, puis distances croissantes.
            self.episode_progress+=float(move)
            lap_finished=bool(self.curriculum_stage==4 and self._lap_crossed
                and self.episode_progress>=self.track.length and not self._lap_invalid
                and not offtrack and not collision)
            invalid_lap=bool(self.curriculum_stage==4 and self._lap_invalid)
            milestone=bool(self.goal_m is not None and self.episode_progress>=self.goal_m
                           and not offtrack and not collision)
            if self.curriculum_stage==1:
                reward=float(2.5*np.clip(move,-3.,5.)-.02
                             -.006*abs(float(action[0])-float(self.last_action[0]))
                             -self.offtrack_penalty*offtrack-25.*collision-5.*idle-2.*wrong_way
                             +20.*milestone)
            else:
                reward=float(np.clip(move,-5.,8.)-.02
                             -.015*abs(float(o['state'][4]))
                             -.008*abs(float(action[0])-float(self.last_action[0]))
                             -self.offtrack_penalty*offtrack-35.*collision-8.*idle-2.*wrong_way
                             +30.*milestone+70.*(lap_finished and not offtrack and not collision))
            reward-=self.offtrack_penalty*invalid_lap
            terminated=bool(offtrack or collision or idle or milestone or lap_finished or invalid_lap)
            time_limit = bool(now-self.t0>=self.episode_seconds)
            truncated=bool(time_limit or (rollout_cut and not terminated))
            self.last_obs=o;self.raw=raw;self.arc=arc;self.damage=new_damage;self.lap=new_lap
            self.last_action=action.copy();self.steps+=1
            # Enregistrer les commandes reelles et les retours des pedales.
            # Utile pour distinguer exploration PPO et probleme de configuration du jeu.
            sent=dict(self.controller.last_command)
            def telemetry_number(*keys):
                for key in keys:
                    if key in raw and raw[key] is not None:
                        try: return float(raw[key])
                        except (ValueError,TypeError): pass
                return None
            info={'progress_m':float(move),'speed_ms':speed,'offtrack':offtrack,'collision':collision,
                  'offtrack_penalty':float(self.offtrack_penalty*offtrack),
                  'world_position_x':float(raw['world_position_x']),
                  'world_position_y':float(raw['world_position_y']),
                  'yaw':float(raw['yaw']),'lateral_m':float(lateral),'track_width_m':float(width),
                  'lap_finished':lap_finished,'lap_invalidated':invalid_lap,
                  'circuit_length_m':float(self.track.length),'lap_count':new_lap,
                  'lap_count_start':self._start_lap,'idle':idle,'wrong_way':wrong_way,'camera_gap_s':gap,
                  'camera_sequence':seq,'episode_id':self.episode_index,
                  'episode_progress_m':float(self.episode_progress),
                  'milestone_reached':bool(milestone),'goal_distance_m':self.goal_m,
                  'curriculum_stage':self.curriculum_stage,
                  'rollout_cut':bool(rollout_cut and not terminated),
                  'time_limit':time_limit,
                  **sent,
                  'accStatus':telemetry_number('accStatus'),
                  'brakeStatus':telemetry_number('brakeStatus'),
                  'rpm':telemetry_number('rpm','RPM','engineRPM'),
                  'gear':telemetry_number('actualGear','gear','Gear','currentGear','CurrentGear')}
            if terminated or truncated: self.safe_stop()
            if self.log_file:
                with self.log_file.open('a',encoding='utf8') as f:
                    f.write(json.dumps({'time':time.time(),'reward':reward,'terminated':terminated,
                             'truncated':truncated,**info},ensure_ascii=False)+'\n')
            return o,reward,terminated,truncated,info
        except (OSError,RuntimeError,TimeoutError,ConnectionError,ValueError) as exc:
            self.safe_stop()
            raise RuntimeError('Incident capteur ou observation : '+str(exc)) from exc
    def close(self):
        # Even if the brake command fails, release UDP, vJoy and shared memory.
        errors=[]
        try:self.safe_stop()
        except Exception as exc:errors.append(str(exc))
        if self.sock is not None:
            try:close_telemetry(self.sock)
            except Exception as exc:errors.append(str(exc))
            finally:self.sock=None
        if self.controller is not None:
            try:self.controller.close()
            except Exception as exc:errors.append(str(exc))
            finally:self.controller=None
        for name in ('dual_shm','meta_shm'):
            mapping=getattr(getattr(self.camera,'cam',None),name,None)
            if mapping is not None:
                try:mapping.close()
                except Exception as exc:errors.append(str(exc))
        if errors:raise RuntimeError('Nettoyage evaluation : '+'; '.join(errors))